# Ozon E-CUP 2026 — E5-small Restore 30k → Full Val → Export

Этот ноутбук **не обучает модель заново**.

Он предназначен для сохранённого checkpoint:

- `intfloat/multilingual-e5-small`
- step `30000`
- fast balanced LLM group holdout Macro PR-AUC ≈ `0.786379`
- `MAX_LEN = 192`

Что делает notebook:

1. Находит `e5_macro_v2_best.pt` **или** `e5_macro_v2_best.zip` в `/kaggle/input` / `/kaggle/working`.
2. Проверяет metadata checkpoint.
3. В точности восстанавливает group split и preprocessing из train-ноутбука.
4. Не загружает тексты всех 13.4M товаров в RAM: из `items.parquet` сохраняет только товары, нужные для validation.
5. Восстанавливает `BertForSequenceClassification` из checkpoint.
6. Сначала воспроизводит fast validation — ожидаем число близкое к `0.786379`.
7. Затем считает **FULL LLM GROUP HOLDOUT Macro PR-AUC** и таблицу по 20 категориям.
8. Считает manual group holdout только как diagnostic.
9. Экспортирует модель и tokenizer в Hugging Face format.
10. Сохраняет `metrics.json`, per-category CSV и ZIP модели в `/kaggle/working`.

## Перед запуском

- Добавь основной dataset с файлами `items.parquet`, `items_human.parquet`, `matches.parquet`, `matches_llm.parquet`.
- Добавь checkpoint как Kaggle Dataset. Имя файла может быть `.pt` или `.zip`: `torch.load` умеет читать оба, потому что PyTorch checkpoint сам использует ZIP-container внутри.
- Включи GPU.
- Включи Internet, чтобы один раз получить config/tokenizer `intfloat/multilingual-e5-small`.

Сам notebook **не содержит Kaggle-specific metadata** и не меняет Accelerator/Internet в Settings.

In [1]:
import os
import gc
import re
import json
import math
import time
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# -----------------------------
# Data paths
# -----------------------------
BASE = "/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items"

ITEMS_PATH = f"{BASE}/items.parquet"
ITEMS_HUMAN_PATH = f"{BASE}/items_human.parquet"
MATCHES_PATH = f"{BASE}/matches.parquet"
MATCHES_LLM_PATH = f"{BASE}/matches_llm.parquet"

# -----------------------------
# Expected checkpoint metadata
# -----------------------------
EXPECTED_MODEL_NAME = "intfloat/multilingual-e5-small"
EXPECTED_STEP = 30000
EXPECTED_FAST_METRIC = 0.7863788901130114
EXPECTED_MAX_LEN = 192

# Exact preprocessing parameters from training
MAX_ATTR_CHARS = 460

# Exact split parameters from training
LLM_VAL_FRAC = 0.03
LLM_VAL_SEED = 13
MANUAL_VAL_FRAC = 0.20
MANUAL_VAL_SEED = 42
FAST_VAL_PER_CATEGORY = 3000
FAST_VAL_SAMPLE_SEED = 0

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

assert torch.cuda.is_available(), "Включи GPU Accelerator в Kaggle."
device = torch.device("cuda")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

# Inference only: safe defaults.
if gpu_mem_gb >= 35:
    PRED_BATCH = 192
elif gpu_mem_gb >= 14:
    PRED_BATCH = 64
else:
    PRED_BATCH = 32

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("GPU:", gpu_name)
print(f"GPU memory: {gpu_mem_gb:.1f} GB")
print("prediction batch:", PRED_BATCH)

GPU: Tesla T4
GPU memory: 14.6 GB
prediction batch: 64


## 1. Проверяем dataset paths и автоматически находим checkpoint

Если checkpoint был загружен на Kaggle как отдельный Dataset, его путь обычно будет вида:

`/kaggle/input/<dataset-name>/e5_macro_v2_best.zip`

Поиск идёт только по точным ожидаемым именам.

In [2]:
import os
import zipfile
from pathlib import Path

# Проверяем основные данные
for p in [ITEMS_PATH, ITEMS_HUMAN_PATH, MATCHES_PATH, MATCHES_LLM_PATH]:
    assert os.path.exists(p), f"Не найден data file: {p}"
    print(f"{os.path.basename(p):24s} {os.path.getsize(p)/1024**3:8.3f} GB")


# Точный путь к Kaggle Dataset с checkpoint
CKPT_DATASET_ROOT = Path(
    "/kaggle/input/datasets/mihailivanovvvv/"
    "first-model-ozon-ecup-torch-checpoint"
)

assert CKPT_DATASET_ROOT.exists(), (
    f"Dataset с checkpoint не найден: {CKPT_DATASET_ROOT}"
)

print("\nCheckpoint dataset contents:")
for p in CKPT_DATASET_ROOT.rglob("*"):
    if p.is_file():
        print(" -", p.relative_to(CKPT_DATASET_ROOT))


# ---------------------------------------------------------
# Вариант 1: Kaggle оставил исходный .pt / .zip как файл
# ---------------------------------------------------------
file_candidates = []

for name in [
    "e5_macro_v2_best.pt",
    "e5_macro_v2_best.zip",
]:
    file_candidates.extend(
        CKPT_DATASET_ROOT.rglob(name)
    )

if file_candidates:
    CKPT_PATH = str(file_candidates[0])

    print("\nFound original checkpoint file:")
    print(CKPT_PATH)
    print(
        f"Size: {os.path.getsize(CKPT_PATH) / 1024**2:.1f} MB"
    )


# ---------------------------------------------------------
# Вариант 2: Kaggle распаковал torch.save ZIP
# ---------------------------------------------------------
else:
    print(
        "\nИсходного .pt/.zip файла нет — "
        "похоже, Kaggle распаковал checkpoint."
    )

    # Ищем папку, содержащую data.pkl.
    data_pkl_files = list(
        CKPT_DATASET_ROOT.rglob("data.pkl")
    )

    assert data_pkl_files, (
        "Не найден ни checkpoint-файл, ни data.pkl. "
        "Проверь содержимое Dataset выше."
    )

    extracted_root = data_pkl_files[0].parent

    print("Detected extracted checkpoint root:")
    print(extracted_root)

    # PyTorch serialization ожидает общий top-level prefix.
    reconstructed_path = Path(
        "/kaggle/working/e5_macro_v2_best_repacked.pt"
    )

    prefix = "e5_macro_v2_best"

    with zipfile.ZipFile(
        reconstructed_path,
        mode="w",
        compression=zipfile.ZIP_STORED,
    ) as zf:
        for file_path in extracted_root.rglob("*"):
            if not file_path.is_file():
                continue

            relative = file_path.relative_to(extracted_root)

            # Восстанавливаем структуру вида:
            # e5_macro_v2_best/data.pkl
            # e5_macro_v2_best/data/0
            # ...
            arcname = Path(prefix) / relative

            zf.write(
                file_path,
                arcname=str(arcname),
            )

    CKPT_PATH = str(reconstructed_path)

    print("\nReconstructed checkpoint:")
    print(CKPT_PATH)
    print(
        f"Size: {os.path.getsize(CKPT_PATH) / 1024**2:.1f} MB"
    )


print("\nUsing checkpoint:")
print(CKPT_PATH)

items.parquet               3.822 GB
items_human.parquet         0.199 GB
matches.parquet             0.004 GB
matches_llm.parquet         0.098 GB

Checkpoint dataset contents:
 - e5_macro_v2_best/.format_version
 - e5_macro_v2_best/.storage_alignment
 - e5_macro_v2_best/data.pkl
 - e5_macro_v2_best/version
 - e5_macro_v2_best/byteorder
 - e5_macro_v2_best/.data/serialization_id
 - e5_macro_v2_best/data/7
 - e5_macro_v2_best/data/135
 - e5_macro_v2_best/data/47
 - e5_macro_v2_best/data/183
 - e5_macro_v2_best/data/17
 - e5_macro_v2_best/data/81
 - e5_macro_v2_best/data/19
 - e5_macro_v2_best/data/199
 - e5_macro_v2_best/data/121
 - e5_macro_v2_best/data/192
 - e5_macro_v2_best/data/22
 - e5_macro_v2_best/data/2
 - e5_macro_v2_best/data/164
 - e5_macro_v2_best/data/147
 - e5_macro_v2_best/data/145
 - e5_macro_v2_best/data/137
 - e5_macro_v2_best/data/35
 - e5_macro_v2_best/data/92
 - e5_macro_v2_best/data/50
 - e5_macro_v2_best/data/23
 - e5_macro_v2_best/data/87
 - e5_macro_v2_best/da

## 2. Проверяем содержимое checkpoint

Файл с расширением `.zip` **не нужно распаковывать**.  
Это обычный результат `torch.save(...)`, и PyTorch сам хранит его во внутреннем ZIP-format.

In [3]:
t0 = time.time()

ckpt = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False,
)

assert isinstance(ckpt, dict), type(ckpt)

print("keys:", ckpt.keys())
print("model_name:", ckpt.get("model_name"))
print("opt_step:", ckpt.get("opt_step"))
print("metric:", ckpt.get("metric"))
print("max_len:", ckpt.get("max_len"))
print("model tensors:", len(ckpt["model"]))
print(f"loaded in {time.time()-t0:.1f}s")

assert ckpt["model_name"] == EXPECTED_MODEL_NAME
assert ckpt["opt_step"] == EXPECTED_STEP
assert ckpt["max_len"] == EXPECTED_MAX_LEN
assert "classifier.weight" in ckpt["model"]
assert "classifier.bias" in ckpt["model"]

MODEL_NAME = ckpt["model_name"]
MAX_LEN = int(ckpt["max_len"])

print("\nCheckpoint metadata OK.")

keys: dict_keys(['model', 'metric', 'opt_step', 'model_name', 'max_len', 'category_weight'])
model_name: intfloat/multilingual-e5-small
opt_step: 30000
metric: 0.7863788901130114
max_len: 192
model tensors: 201
loaded in 0.3s

Checkpoint metadata OK.


## 3. В точности восстанавливаем group splits

Это та же union-find функция и те же seed/frac, что использовались при training.

Важно:
- train заново не нужен;
- для LLM validation сохраняем только confident labels `target <= 0.2` или `target >= 0.8`;
- затем бинаризуем их **только для метрики**;
- manual validation нужен только как diagnostic.

In [4]:
def group_val_mask(df, val_frac, seed):
    parent = {}

    def find(x):
        p = parent.setdefault(x, x)
        while p != parent[p]:
            parent[p] = parent[parent[p]]
            p = parent[p]
        parent[x] = p
        return p

    for a, b in zip(df.id1.values, df.id2.values):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    comp = np.fromiter(
        (find(i) for i in df.id1.values),
        dtype=np.int64,
        count=len(df),
    )

    rng = np.random.RandomState(seed)
    uniq = np.unique(comp)
    val_set = set(uniq[rng.rand(len(uniq)) < val_frac].tolist())

    return np.fromiter(
        (c in val_set for c in comp),
        dtype=bool,
        count=len(df),
    )

t0 = time.time()

manual_all = pd.read_parquet(MATCHES_PATH)
manual_val_mask = group_val_mask(
    manual_all,
    MANUAL_VAL_FRAC,
    MANUAL_VAL_SEED,
)
manual_val = manual_all[manual_val_mask].copy()

llm_all = pd.read_parquet(MATCHES_LLM_PATH)
llm_val_mask = group_val_mask(
    llm_all,
    LLM_VAL_FRAC,
    LLM_VAL_SEED,
)

llm_val_raw = llm_all[llm_val_mask].copy()
llm_val = llm_val_raw[
    (llm_val_raw.target <= 0.2) |
    (llm_val_raw.target >= 0.8)
].copy()
llm_val["target"] = (llm_val["target"] >= 0.5).astype(np.int8)

print("manual all:", len(manual_all))
print("manual val:", len(manual_val))
print("llm all:", len(llm_all))
print("llm val raw:", len(llm_val_raw))
print("llm val confident:", len(llm_val))
print(f"split time: {(time.time()-t0)/60:.2f} min")

# Free train-side objects; we only validate in this notebook.
del manual_all, manual_val_mask
del llm_all, llm_val_mask, llm_val_raw
gc.collect()

manual all: 365654
manual val: 72948
llm all: 11187780
llm val raw: 237386
llm val confident: 191555
split time: 0.80 min


160

## 4. Собираем только нужные товары

В training notebook мы строили text для всех 13.4M товаров — это занимало ~27 минут и много RAM.

Для validation достаточно товаров, которые встречаются в:
- full LLM holdout;
- manual holdout.

Поэтому один раз сканируем `items.parquet`, но сохраняем только нужные IDs.

In [5]:
needed_ids = set(manual_val.id1.values.tolist())
needed_ids.update(manual_val.id2.values.tolist())
needed_ids.update(llm_val.id1.values.tolist())
needed_ids.update(llm_val.id2.values.tolist())

print("unique items required:", f"{len(needed_ids):,}")

unique items required: 394,813


### Тот же preprocessing V2, что был при training

Особенно важно сохранить:
- lower-case;
- `ё → е`;
- category в тексте;
- тот же порядок priority attributes;
- `MAX_ATTR_CHARS = 460`;
- `MAX_LEN = 192`.

In [6]:
SPACE_RE = re.compile(r"\s+")
MULTIPLY_RE = re.compile(r"[×хХ]")

KEY_ORDER = [
    "бренд", "brand",
    "артикул", "партномер", "part number", "partnumber", "oem",
    "код", "sku", "модель", "model",
    "размер", "size", "рост", "обхват", "пол", "gender",
    "цвет", "color", "материал", "material", "сезон",
    "объем", "обьем", "volume", "вес", "weight",
    "длина", "ширина", "высота",
    "количество", "комплектация", "упаков",
    "тип", "type",
]

def normalize_piece(x):
    if x is None:
        return ""
    s = str(x).lower().replace("ё", "е")
    s = MULTIPLY_RE.sub("x", s)
    s = s.replace(",", ".")
    s = SPACE_RE.sub(" ", s).strip()
    return s

def safe_attrs(attributes):
    if isinstance(attributes, dict):
        obj = attributes
    elif isinstance(attributes, str):
        try:
            obj = json.loads(attributes)
        except Exception:
            obj = {}
    else:
        obj = {}

    if not isinstance(obj, dict):
        return {}

    out = {}
    for k, v in obj.items():
        kk = normalize_piece(k)
        vv = normalize_piece(v)
        if kk and vv:
            out[kk] = vv
    return out

def build_text_v2(name, attributes, category, max_attr_chars=MAX_ATTR_CHARS):
    cat = normalize_piece(category)
    nm = normalize_piece(name)
    attrs = safe_attrs(attributes)

    picked = []
    used = set()

    for want in KEY_ORDER:
        for k, v in attrs.items():
            if k in used:
                continue
            if want in k:
                picked.append(f"{k}: {v}")
                used.add(k)

    rest = [f"{k}: {v}" for k, v in attrs.items() if k not in used]
    attr_text = " ; ".join(picked + rest)[:max_attr_chars]

    return f"категория: {cat} | название: {nm} | атрибуты: {attr_text}"

In [7]:
t0 = time.time()

item_text = {}
item_cat = {}

pf = pq.ParquetFile(ITEMS_PATH)

for batch_id, batch in enumerate(
    pf.iter_batches(
        columns=["id", "name", "attributes", "category"],
        batch_size=400_000,
    ),
    start=1,
):
    pdf = batch.to_pandas()

    mask = pdf["id"].isin(needed_ids)
    sub = pdf.loc[mask]

    for i, n, a, c in sub.itertuples(index=False, name=None):
        item_text[i] = build_text_v2(n, a, c)
        item_cat[i] = c

    if batch_id % 5 == 0:
        print(
            f"batches={batch_id:3d} "
            f"found={len(item_text):,}/{len(needed_ids):,} "
            f"time={time.time()-t0:.0f}s",
            flush=True,
        )

    del pdf, sub, batch
    gc.collect()

missing = needed_ids - set(item_text)

print(f"\nLoaded required items: {len(item_text):,}")
print("missing required items:", len(missing))
print(f"scan time: {(time.time()-t0)/60:.2f} min")

assert not missing, (
    f"Не найдены {len(missing)} товаров, validation будет некорректен."
)

manual_val["category"] = [
    item_cat[i] for i in manual_val.id1.values
]
llm_val["category"] = [
    item_cat[i] for i in llm_val.id1.values
]

print("\nReal validation categories:")
print(sorted(llm_val["category"].unique()))

batches=  5 found=57,087/394,813 time=35s
batches= 10 found=113,785/394,813 time=57s
batches= 15 found=172,958/394,813 time=80s
batches= 20 found=225,925/394,813 time=107s
batches= 25 found=287,361/394,813 time=135s
batches= 30 found=354,614/394,813 time=164s

Loaded required items: 394,813
missing required items: 0
scan time: 3.10 min

Real validation categories:
['Автотовары', 'Аптека', 'Бытовая техника', 'Бытовая химия', 'Галантерея и аксессуары', 'Детские товары', 'Дом и сад', 'Канцелярские товары', 'Красота и гигиена', 'Мебель', 'Музыкальные инструменты', 'Обувь', 'Одежда', 'Продукты питания', 'Спорт и отдых', 'Строительство и ремонт', 'Товары для животных', 'Хобби и творчество', 'Электроника', 'Ювелирные изделия']


## 5. Восстанавливаем tokenizer и модель из checkpoint

Мы не скачиваем pretrained weights заново:
- с Hugging Face получаем только config + tokenizer;
- модель создаём из config;
- **все 201 tensors** перезаписываем сохранённым state dict;
- `strict=True` гарантирует, что ничего не потерялось.

`classifier.weight` и `classifier.bias` уже есть в checkpoint — это именно обученный head.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=1,
)

model = AutoModelForSequenceClassification.from_config(config)

load_result = model.load_state_dict(
    ckpt["model"],
    strict=True,
)

print(load_result)

model = model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params/1e6:.1f}M")
print("model restored from step:", ckpt["opt_step"])
print("checkpoint fast metric:", ckpt["metric"])

# No longer need a second CPU copy of the model tensors.
del ckpt
gc.collect()

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

<All keys matched successfully>
parameters: 117.7M
model restored from step: 30000
checkpoint fast metric: 0.7863788901130114


190

## 6. Dataset, inference и Macro PR-AUC

Prediction code использует pair tokenization `(text1, text2)` с тем же `MAX_LEN`.

Если T4 даст CUDA OOM, снизь `PRED_BATCH` с `64` до `48` или `32` и перезапусти эту ячейку.

In [9]:
class PairDataset(Dataset):
    def __init__(self, pairs_df):
        self.id1 = pairs_df.id1.values
        self.id2 = pairs_df.id2.values
        self.y = pairs_df.target.values.astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        a = self.id1[idx]
        b = self.id2[idx]
        return item_text[a], item_text[b], self.y[idx]

def collate_eval(batch):
    t1, t2, y = zip(*batch)

    enc = tokenizer(
        list(t1),
        list(t2),
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )

    return enc, torch.tensor(y, dtype=torch.float32)

@torch.inference_mode()
def predict(model, pairs_df, batch_size=PRED_BATCH):
    model.eval()

    dl = DataLoader(
        PairDataset(pairs_df),
        batch_size=batch_size,
        collate_fn=collate_eval,
        num_workers=0,
        shuffle=False,
        pin_memory=True,
    )

    preds = []
    done = 0
    t0 = time.time()

    for batch_idx, (enc, _) in enumerate(dl, start=1):
        enc = {
            k: v.to(device, non_blocking=True)
            for k, v in enc.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = model(**enc).logits.squeeze(-1)

        preds.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )

        done += len(logits)

        if batch_idx % 500 == 0:
            speed = done / max(time.time()-t0, 1e-6)
            print(
                f"{done:,}/{len(pairs_df):,} "
                f"({speed:.0f} pair/s)",
                flush=True,
            )

    return np.concatenate(preds)

def macro_pr_auc(pairs_df, preds):
    z = pairs_df[["category", "target"]].reset_index(drop=True).copy()
    z["pred"] = np.asarray(preds)

    rows = []

    for cat, g in z.groupby("category", observed=True):
        y = g["target"].to_numpy()
        p = g["pred"].to_numpy()

        if y.sum() == 0:
            ap = np.nan
        else:
            ap = average_precision_score(y, p)

        rows.append({
            "category": cat,
            "pairs": len(g),
            "positive_rate": float(y.mean()),
            "PR_AUC": ap,
        })

    table = (
        pd.DataFrame(rows)
        .sort_values("PR_AUC")
        .reset_index(drop=True)
    )

    macro = float(table["PR_AUC"].dropna().mean())
    return macro, table

def make_balanced_fast_val(
    df,
    per_category=FAST_VAL_PER_CATEGORY,
    seed=FAST_VAL_SAMPLE_SEED,
):
    parts = []

    for _, g in df.groupby("category", observed=True):
        n = min(per_category, len(g))
        parts.append(g.sample(n=n, random_state=seed))

    return pd.concat(parts, ignore_index=True)

## 7. Sanity check: воспроизводим fast validation

Это критическая проверка.

Training run сохранил best checkpoint на step 30000 с:

`LLM balanced-fast Macro PR-AUC = 0.7863788901`

Мы восстанавливаем **тот же** balanced-fast subset. Ожидаем близкое число.

Если отличие больше примерно `0.003`, full validation лучше не интерпретировать, пока не найдена причина.

In [10]:
llm_val_fast = make_balanced_fast_val(llm_val)

print("fast validation pairs:", len(llm_val_fast))

t0 = time.time()
fast_pred = predict(model, llm_val_fast)
fast_macro, fast_table = macro_pr_auc(
    llm_val_fast,
    fast_pred,
)

delta = fast_macro - EXPECTED_FAST_METRIC

print("\n" + "=" * 90)
print(f"RESTORED FAST MACRO PR-AUC: {fast_macro:.9f}")
print(f"EXPECTED FROM TRAIN:        {EXPECTED_FAST_METRIC:.9f}")
print(f"DELTA:                      {delta:+.9f}")
print(f"time: {(time.time()-t0)/60:.2f} min")
print("=" * 90)

display(fast_table)

assert abs(delta) < 0.003, (
    "Fast metric не воспроизвелась достаточно близко. "
    "Не доверяй full validation до выяснения причины."
)

print("\nSanity check PASSED.")

fast validation pairs: 56275
32,000/56,275 (540 pair/s)

RESTORED FAST MACRO PR-AUC: 0.786378890
EXPECTED FROM TRAIN:        0.786378890
DELTA:                      +0.000000000
time: 1.80 min


,category,pairs,positive_rate,PR_AUC
0,Обувь,3000,0.069667,0.311605
1,Одежда,3000,0.057667,0.439770
2,Галантерея и аксессуары,3000,0.074667,0.550660
3,Ювелирные изделия,1265,0.181028,0.641862
4,Красота и гигиена,3000,0.205000,0.787648
5,Дом и сад,3000,0.209667,0.800050
6,Канцелярские товары,3000,0.275333,0.810164
7,Спорт и отдых,3000,0.205667,0.816289
8,Электроника,3000,0.089333,0.818594
9,Детские товары,3000,0.297667,0.819411



Sanity check PASSED.


## 8. FULL LLM GROUP HOLDOUT — главная метрика

Это главный результат notebook.

Для командного решения ориентируемся именно на LLM group holdout, а не на manual holdout.

In [11]:
t0 = time.time()

llm_pred = predict(model, llm_val)
llm_macro, llm_table = macro_pr_auc(
    llm_val,
    llm_pred,
)

print("\n" + "=" * 100)
print(f"FULL LLM GROUP HOLDOUT MACRO PR-AUC: {llm_macro:.9f}")
print(f"pairs: {len(llm_val):,}")
print(f"time: {(time.time()-t0)/60:.2f} min")
print("=" * 100)

display(llm_table)

32,000/191,555 (511 pair/s)
64,000/191,555 (508 pair/s)
96,000/191,555 (506 pair/s)
128,000/191,555 (506 pair/s)
160,000/191,555 (505 pair/s)

FULL LLM GROUP HOLDOUT MACRO PR-AUC: 0.786482334
pairs: 191,555
time: 6.34 min


,category,pairs,positive_rate,PR_AUC
0,Обувь,6199,0.072270,0.323267
1,Одежда,10744,0.054263,0.431439
2,Галантерея и аксессуары,11660,0.078302,0.541161
3,Ювелирные изделия,1265,0.181028,0.641862
4,Красота и гигиена,16959,0.201545,0.787953
5,Дом и сад,13003,0.214181,0.794009
6,Электроника,14657,0.088490,0.813678
7,Детские товары,11415,0.304424,0.816749
8,Канцелярские товары,7406,0.276533,0.818162
9,Спорт и отдых,9752,0.203651,0.822123


## 9. Проверяем слабые fashion-категории

При training был найден баг в ручном списке fashion boost:
- реальные названия: `Галантерея и аксессуары`, `Ювелирные изделия`;
- в V2 дополнительный multiplier `1.15` для них не применился.

Это **не ломает checkpoint**: общий macro category balancing работал для всех категорий.  
Здесь просто отдельно фиксируем фактическое качество 4 fashion-категорий перед следующим решением.

In [12]:
REAL_FASHION_CATEGORIES = {
    "Обувь",
    "Одежда",
    "Галантерея и аксессуары",
    "Ювелирные изделия",
}

fashion_table = (
    llm_table[
        llm_table["category"].isin(REAL_FASHION_CATEGORIES)
    ]
    .sort_values("PR_AUC")
    .reset_index(drop=True)
)

print("Fashion categories:")
display(fashion_table)

non_fashion_macro = llm_table.loc[
    ~llm_table["category"].isin(REAL_FASHION_CATEGORIES),
    "PR_AUC",
].mean()

fashion_macro = fashion_table["PR_AUC"].mean()

print("fashion macro:", float(fashion_macro))
print("non-fashion macro:", float(non_fashion_macro))

Fashion categories:


,category,pairs,positive_rate,PR_AUC
0,Обувь,6199,0.072270,0.323267
1,Одежда,10744,0.054263,0.431439
2,Галантерея и аксессуары,11660,0.078302,0.541161
3,Ювелирные изделия,1265,0.181028,0.641862


fashion macro: 0.4844321981475632
non-fashion macro: 0.8619948679370711


## 10. Manual group holdout — только diagnostic

Команда уже проверила, что ручной holdout плохо отражает leaderboard.  
Эту цифру **не используем для выбора checkpoint**.

In [13]:
t0 = time.time()

manual_pred = predict(model, manual_val)
manual_macro, manual_table = macro_pr_auc(
    manual_val,
    manual_pred,
)

print("\n" + "=" * 100)
print(
    "MANUAL GROUP HOLDOUT MACRO PR-AUC "
    f"(diagnostic only): {manual_macro:.9f}"
)
print(f"pairs: {len(manual_val):,}")
print(f"time: {(time.time()-t0)/60:.2f} min")
print("=" * 100)

display(manual_table)

32,000/72,948 (507 pair/s)
64,000/72,948 (500 pair/s)

MANUAL GROUP HOLDOUT MACRO PR-AUC (diagnostic only): 0.677023885
pairs: 72,948
time: 2.43 min


,category,pairs,positive_rate,PR_AUC
0,Обувь,3603,0.102970,0.281817
1,Ювелирные изделия,3721,0.125773,0.358049
2,Одежда,4587,0.119468,0.393088
3,Галантерея и аксессуары,3597,0.174868,0.538207
4,Мебель,3665,0.159345,0.609702
5,Спорт и отдых,3562,0.254632,0.619031
6,Автотовары,3779,0.173062,0.641313
7,Электроника,3767,0.170427,0.648752
8,Канцелярские товары,3494,0.340298,0.688544
9,Строительство и ремонт,3541,0.217170,0.720388


## 11. Экспортируем полноценную модель

После этой ячейки для inference больше не нужен исходный training checkpoint.

Получим:

- `model.safetensors`
- `config.json`
- tokenizer files
- `metrics.json`
- `llm_by_category.csv`
- `manual_by_category.csv`
- `fashion_by_category.csv`

Всё складывается в одну папку и ZIP.

In [14]:
EXPORT_DIR = "/kaggle/working/e5_macro_v2_30k_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Hugging Face model + tokenizer
model_cpu = model.to("cpu")
model_cpu.save_pretrained(
    EXPORT_DIR,
    safe_serialization=True,
)
tokenizer.save_pretrained(EXPORT_DIR)

# Tables
llm_table.to_csv(
    os.path.join(EXPORT_DIR, "llm_by_category.csv"),
    index=False,
)
manual_table.to_csv(
    os.path.join(EXPORT_DIR, "manual_by_category.csv"),
    index=False,
)
fashion_table.to_csv(
    os.path.join(EXPORT_DIR, "fashion_by_category.csv"),
    index=False,
)

metrics = {
    "experiment": "e5-small_macro-llm_stageA_v2_restore30k",
    "model_name": MODEL_NAME,
    "checkpoint_step": EXPECTED_STEP,
    "train_fast_macro_recorded": EXPECTED_FAST_METRIC,
    "restored_fast_macro": float(fast_macro),
    "restored_fast_delta": float(delta),
    "full_llm_group_holdout_macro": float(llm_macro),
    "manual_group_holdout_macro_diagnostic": float(manual_macro),
    "fashion_macro": float(fashion_macro),
    "non_fashion_macro": float(non_fashion_macro),
    "max_len": MAX_LEN,
    "max_attr_chars": MAX_ATTR_CHARS,
    "llm_val_pairs": int(len(llm_val)),
    "manual_val_pairs": int(len(manual_val)),
    "real_fashion_categories": sorted(REAL_FASHION_CATEGORIES),
    "note": (
        "Checkpoint trained on LLM stage-A only. "
        "Manual metric is diagnostic only. "
        "Original V2 fashion extra boost missed exact names for "
        "Galantereya i aksessuary / Yuvelirnye izdeliya, "
        "but general macro category weighting remained active."
    ),
}

with open(
    os.path.join(EXPORT_DIR, "metrics.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metrics,
        f,
        ensure_ascii=False,
        indent=2,
    )

zip_path = shutil.make_archive(
    "/kaggle/working/e5_macro_v2_30k_export",
    "zip",
    root_dir="/kaggle/working",
    base_dir="e5_macro_v2_30k_export",
)

print("EXPORT DIR:", EXPORT_DIR)
print("ZIP:", zip_path)
print(f"ZIP size: {os.path.getsize(zip_path)/1024**3:.3f} GB")
print("\nMETRICS:")
print(json.dumps(metrics, ensure_ascii=False, indent=2))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

EXPORT DIR: /kaggle/working/e5_macro_v2_30k_export
ZIP: /kaggle/working/e5_macro_v2_30k_export.zip
ZIP size: 0.336 GB

METRICS:
{
  "experiment": "e5-small_macro-llm_stageA_v2_restore30k",
  "model_name": "intfloat/multilingual-e5-small",
  "checkpoint_step": 30000,
  "train_fast_macro_recorded": 0.7863788901130114,
  "restored_fast_macro": 0.7863788901130114,
  "restored_fast_delta": 0.0,
  "full_llm_group_holdout_macro": 0.7864823339791696,
  "manual_group_holdout_macro_diagnostic": 0.6770238851603795,
  "fashion_macro": 0.4844321981475632,
  "non_fashion_macro": 0.8619948679370711,
  "max_len": 192,
  "max_attr_chars": 460,
  "llm_val_pairs": 191555,
  "manual_val_pairs": 72948,
  "real_fashion_categories": [
    "Галантерея и аксессуары",
    "Обувь",
    "Одежда",
    "Ювелирные изделия"
  ],
  "note": "Checkpoint trained on LLM stage-A only. Manual metric is diagnostic only. Original V2 fashion extra boost missed exact names for Galantereya i aksessuary / Yuvelirnye izdeliya, but

## 12. Создаём короткий run summary для GitHub / командного README

Файл сохраняется в `/kaggle/working/e5_macro_v2_30k_RUN_SUMMARY.md`.

Его можно использовать при коммите результатов эксперимента.

In [15]:
summary = f"""# E5-small Macro LLM Stage-A V2 — checkpoint 30k

## Model

- Backbone: `{MODEL_NAME}`
- Training distribution: LLM pairs only
- Checkpoint: optimizer step `{EXPECTED_STEP}`
- MAX_LEN: `{MAX_LEN}`
- Attribute character budget: `{MAX_ATTR_CHARS}`
- Random pair swap during training: 0.5
- Category-balanced soft BCE
- Confidence-weighted LLM soft labels
- Manual stage-B: **not used**

## Validation

- Recorded balanced-fast LLM Macro PR-AUC during training: `{EXPECTED_FAST_METRIC:.6f}`
- Restored balanced-fast LLM Macro PR-AUC: `{fast_macro:.6f}`
- **Full LLM group holdout Macro PR-AUC: `{llm_macro:.6f}`**
- Manual group holdout Macro PR-AUC (diagnostic only): `{manual_macro:.6f}`
- Fashion subset mean PR-AUC: `{fashion_macro:.6f}`

## Known issue in this version

The original V2 `FASHION_CATEGORIES` list used incorrect exact strings for two categories.
Therefore the extra `1.15` fashion multiplier did not apply to:

- `Галантерея и аксессуары`
- `Ювелирные изделия`

The general category balancing still applied to every category.

## Export

Kaggle artifact:

`/kaggle/working/e5_macro_v2_30k_export.zip`
"""

summary_path = "/kaggle/working/e5_macro_v2_30k_RUN_SUMMARY.md"

with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary)

print(summary)
print("\nSaved:", summary_path)

# E5-small Macro LLM Stage-A V2 — checkpoint 30k

## Model

- Backbone: `intfloat/multilingual-e5-small`
- Training distribution: LLM pairs only
- Checkpoint: optimizer step `30000`
- MAX_LEN: `192`
- Attribute character budget: `460`
- Random pair swap during training: 0.5
- Category-balanced soft BCE
- Confidence-weighted LLM soft labels
- Manual stage-B: **not used**

## Validation

- Recorded balanced-fast LLM Macro PR-AUC during training: `0.786379`
- Restored balanced-fast LLM Macro PR-AUC: `0.786379`
- **Full LLM group holdout Macro PR-AUC: `0.786482`**
- Manual group holdout Macro PR-AUC (diagnostic only): `0.677024`
- Fashion subset mean PR-AUC: `0.484432`

## Known issue in this version

The original V2 `FASHION_CATEGORIES` list used incorrect exact strings for two categories.
Therefore the extra `1.15` fashion multiplier did not apply to:

- `Галантерея и аксессуары`
- `Ювелирные изделия`

The general category balancing still applied to every category.

## Export

Kaggle art

# Готово

После успешного выполнения сохрани/скачай из Kaggle Output:

1. `e5_macro_v2_30k_export.zip` — модель для inference/submission.
2. `e5_macro_v2_30k_RUN_SUMMARY.md` — результаты для GitHub/README.

Перед сборкой submit сначала сравни:
- `full_llm_group_holdout_macro`;
- per-category PR-AUC;
- fashion categories;
- текущий командный reference.

Следующий этап — адаптировать `run.py` контейнера под экспортированную E5-small, измерить inference на тестовом размере и только после этого тратить командный submit.